In [ ]:
# ============================================================
# 0. IMPORTS Y CONFIGURACIÓN
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_columns', None)


# ============================================================
# 1. CARGA DE DATASETS
# ============================================================

demo = pd.read_csv("df_final_demo.csv")
web_data_pt_1 = pd.read_csv("df_final_web_data_pt_1.csv")
web_data_pt_2 = pd.read_csv("df_final_web_data_pt_2.csv")
experiment_clients = pd.read_csv("df_final_experiment_clients.csv")


# ============================================================
# 2. UNIÓN DE WEB_DATA (pt1 + pt2)
# ============================================================

df_web_union = pd.concat([web_data_pt_1, web_data_pt_2], ignore_index=True)


# ============================================================
# 3. ESTANDARIZAR NOMBRES DE COLUMNAS
# ============================================================

demo.columns = demo.columns.str.lower().str.strip()
experiment_clients.columns = experiment_clients.columns.str.lower().str.strip()
df_web_union.columns = df_web_union.columns.str.lower().str.strip()


# ============================================================
# 4. LIMPIEZA DE DATASETS
# ============================================================

# ---- DEMO ----
demo.dropna(inplace=True)
demo['gendr'] = demo['gendr'].replace('X', 'U')
demo = demo[demo['clnt_age'] >= 18].copy()

cols_int = ['clnt_tenure_yr','clnt_tenure_mnth','clnt_age',
            'num_accts','calls_6_mnth','logons_6_mnth']
for col in cols_int:
    demo[col] = demo[col].astype(int)

# ---- EXPERIMENTO ----
experiment_clients = experiment_clients.dropna(subset=['variation']).copy()

# ---- WEB ----
df_web_union.drop_duplicates(inplace=True)
df_web_union['date_time'] = pd.to_datetime(df_web_union['date_time'])


# ============================================================
# 5. UNIÓN DE LOS 3 DATASETS
# ============================================================

df_merged = df_web_union.merge(experiment_clients, on='client_id', how='inner')
df_final = df_merged.merge(demo, on='client_id', how='left')
df_final['date_time'] = pd.to_datetime(df_final['date_time'])


# ============================================================
# 6. ORDEN LÓGICO DE PASOS
# ============================================================

ORDEN_PASOS = ['start','step_1','step_2','step_3','confirm']


# ============================================================
# 7. KPI 1 — COMPLETION FLAG
# ============================================================

clientes_confirm = df_final[df_final['process_step']=='confirm']['client_id'].unique()
df_final['completion_flag'] = df_final['client_id'].isin(clientes_confirm).astype(int)


# ============================================================
# 8. KPI 2 — FUNNEL + DROP-OFF
# ============================================================

funnel = (
    df_final.groupby(['variation','process_step'])['client_id']
    .nunique()
    .reset_index()
    .rename(columns={'client_id':'clientes'})
)

funnel['process_step'] = pd.Categorical(funnel['process_step'], categories=ORDEN_PASOS, ordered=True)
funnel = funnel.sort_values(['variation','process_step'])

funnel['drop_off'] = funnel.groupby('variation')['clientes'].transform(
    lambda x: (1 - x/x.shift(1)).round(4)
)

df_final = df_final.merge(
    funnel[['variation','process_step','clientes','drop_off']],
    on=['variation','process_step'],
    how='left'
)


# ============================================================
# 9. KPI 3 — TIEMPO ENTRE PASOS
# ============================================================

df_final = df_final.sort_values(['visit_id','date_time'])

df_final['time_diff_sec'] = (
    df_final.groupby('visit_id')['date_time']
    .diff()
    .dt.total_seconds()
)

df_final['total_time_visit'] = df_final.groupby('visit_id')['time_diff_sec'].transform('sum')
df_final['total_time_client'] = df_final.groupby('client_id')['time_diff_sec'].transform('sum')


# ============================================================
# 10. KPI 4 — BACKTRACKING
# ============================================================

step_order = {'start':1,'step_1':2,'step_2':3,'step_3':4,'confirm':5}
df_final['step_number'] = df_final['process_step'].map(step_order)

df_final['previous_step'] = df_final.groupby('visit_id')['step_number'].shift()
df_final['backtrack'] = (df_final['step_number'] < df_final['previous_step']).astype(int)

df_final['backtrack_count'] = df_final.groupby('client_id')['backtrack'].transform('sum')
df_final['confused_user'] = (df_final['backtrack_count'] > 1).astype(int)


# ============================================================
# 11. EXPORTAR DATASET FINAL
# ============================================================

df_final.to_csv(
    "df_final.csv",
    index=False,
    date_format="%Y-%m-%d %H:%M:%S"
)

print("df_final.csv generado correctamente.")
